# Uji Signifikansi Akurasi: BlazePose vs YOLOv8-pose (Paired t-test)

Menguji apakah perbedaan akurasi sudut (error terhadap ground truth) kedua model **signifikan secara statistik** atau hanya kebetulan.

- Error per frame = |sudut model - ground truth| pada 15 frame GT.
- Uji berpasangan (paired) karena kedua model diukur pada frame yang sama.
- Cek normalitas selisih dengan Shapiro-Wilk; jika normal pakai paired t-test, jika tidak pakai Wilcoxon signed-rank.
- H0: tidak ada perbedaan akurasi. p < 0,05 berarti perbedaan signifikan.

Install bila perlu: `pip install scipy pandas numpy`


In [5]:
import pandas as pd, numpy as np
from scipy import stats

# ====== KONFIG: per kondisi, path frames CSV + ground truth (rata-rata 3x) ======
BASE  = r"C:\Users\ASUS\OneDrive\Documents\Claude\Projects\Seputar Tugas Akhir\Dataset video Bicep curl\comparison_outputs_v3\output dengan format pengukuran GT baru"
SWING = r"C:\Users\ASUS\OneDrive\Documents\Claude\Projects\Seputar Tugas Akhir\Dataset video Bicep curl\comparison_outputs_v3_fixed\output dengan format pengukuran GT baru\output subjek 1 swing curl"

SUBJEK = {
  "Subjek 1 normal":  (BASE + r"\output subjek 1", {140:127.236,166:163.957,192:83.302,215:48.523,352:73.160,391:160.454,411:111.959,421:83.064,447:47.348,465:68.011,636:167.115,667:84.893,819:46.790,839:70.759,905:116.169}),
  "Subjek 2 normal":  (BASE + r"\output subjek 2", {69:161.666,85:102.784,90:69.337,100:37.096,131:97.180,152:166.546,256:59.053,333:75.079,344:42.200,454:107.277,470:142.305,492:58.355,566:78.275,579:38.503,608:106.802}),
  "Swinging":         (SWING, {114:150.345,145:86.074,164:40.913,192:101.357,246:65.457,298:135.228,330:90.556,354:39.123,380:106.916,448:74.831,506:131.464,538:86.687,562:42.480,655:66.163,869:100.794}),
  "Robust": (BASE + r"\output subjek 1 robust test", {108:168.023,132:76.749,148:41.366,169:100.176,223:56.811,289:166.924,313:80.494,333:39.947,357:96.747,419:65.721,487:170.185,513:78.460,615:38.826,691:61.790,733:105.366}),
  "Subjek 3 normal":  (BASE + r"\output subjek 3", {141:156.060,172:77.456,192:34.094,282:62.531,324:99.303,342:147.448,373:83.525,395:38.290,484:68.001,523:108.017,544:154.615,577:80.339,598:41.790,690:64.440,732:103.260}),
  "Occlusion (siku)":  (BASE + r"\output subjek 1 occlusion test", {113:165.057,159:39.218,312:82.572,330:40.979,355:108.225,403:85.997,409:65.261,507:61.792,545:112.782,571:164.272,600:79.325,604:67.887,618:40.747,643:120.403,687:157.061}),
}


In [6]:
def load(folder):
    bp = pd.read_csv(folder + r'\frames_blazepose.csv'); yo = pd.read_csv(folder + r'\frames_yolov8pose.csv')
    return dict(zip(bp['frame'],bp['angle_smooth'])), dict(zip(yo['frame'],yo['angle_smooth']))

all_bp=[]; all_yo=[]
for nama,(folder,gt) in SUBJEK.items():
    bp,yo = load(folder)
    # hanya frame yang KEDUA model deteksi (angle != 0), konsisten dgn kode komparasi
    fr = [f for f in gt if bp.get(f,0)!=0 and yo.get(f,0)!=0]
    ebp=np.array([abs(bp[f]-gt[f]) for f in fr]); eyo=np.array([abs(yo[f]-gt[f]) for f in fr])
    all_bp+=list(ebp); all_yo+=list(eyo); diff=ebp-eyo
    sh=stats.shapiro(diff).pvalue; t=stats.ttest_rel(ebp,eyo); w=stats.wilcoxon(ebp,eyo)
    pakai='t-test' if sh>0.05 else 'Wilcoxon'; pv=t.pvalue if sh>0.05 else w.pvalue
    print(f'===== {nama} (n={len(fr)}) =====')
    print(f'  MAE BlazePose {ebp.mean():.2f} vs YOLOv8 {eyo.mean():.2f} (selisih {ebp.mean()-eyo.mean():+.2f})')
    print(f'  Shapiro p={sh:.3f} -> pakai {pakai} | t-test p={t.pvalue:.4f} | Wilcoxon p={w.pvalue:.4f}')
    print(f'  KESIMPULAN: ' + ('SIGNIFIKAN' if pv<0.05 else 'tidak signifikan') + '\n')

ebp=np.array(all_bp); eyo=np.array(all_yo); diff=ebp-eyo
sh=stats.shapiro(diff).pvalue; t=stats.ttest_rel(ebp,eyo); w=stats.wilcoxon(ebp,eyo)
pakai='t-test' if sh>0.05 else 'Wilcoxon'; pv=t.pvalue if sh>0.05 else w.pvalue
print(f'===== GABUNGAN semua kondisi (n={len(ebp)}) =====')
print(f'  MAE BlazePose {ebp.mean():.2f} vs YOLOv8 {eyo.mean():.2f} | Shapiro p={sh:.3f} -> pakai {pakai}')
print(f'  t-test p={t.pvalue:.4f} | Wilcoxon p={w.pvalue:.4f} -> ' + ('SIGNIFIKAN' if pv<0.05 else 'tidak signifikan'))


===== Subjek 1 normal (n=15) =====
  MAE BlazePose 8.41 vs YOLOv8 8.65 (selisih -0.24)
  Shapiro p=0.747 -> pakai t-test | t-test p=0.8579 | Wilcoxon p=0.8469
  KESIMPULAN: tidak signifikan

===== Subjek 2 normal (n=15) =====
  MAE BlazePose 8.71 vs YOLOv8 12.15 (selisih -3.44)
  Shapiro p=0.069 -> pakai t-test | t-test p=0.0179 | Wilcoxon p=0.0215
  KESIMPULAN: SIGNIFIKAN

===== Swinging (n=15) =====
  MAE BlazePose 8.18 vs YOLOv8 12.10 (selisih -3.92)
  Shapiro p=0.019 -> pakai Wilcoxon | t-test p=0.0652 | Wilcoxon p=0.0946
  KESIMPULAN: tidak signifikan

===== Robust (n=14) =====
  MAE BlazePose 8.02 vs YOLOv8 9.98 (selisih -1.96)
  Shapiro p=0.436 -> pakai t-test | t-test p=0.0913 | Wilcoxon p=0.1353
  KESIMPULAN: tidak signifikan

===== Subjek 3 normal (n=15) =====
  MAE BlazePose 11.62 vs YOLOv8 12.67 (selisih -1.05)
  Shapiro p=0.105 -> pakai t-test | t-test p=0.4030 | Wilcoxon p=0.3894
  KESIMPULAN: tidak signifikan

===== Occlusion (siku) (n=15) =====
  MAE BlazePose 9.98 vs Y